In [1]:
# Install required libraries
!pip install nltk scikit-learn

In [3]:
import nltk
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [4]:
# Import libraries
import nltk
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [5]:
faq_data = {
    "How are you?":"I am doing well. Thanks for asking!",
    "What can you do?": "I can answer frequently asked questions using AI techniques.",
    "What is the weather today?": "I cannot see the weather unless connected to live data.",
    "What can I ask you about?": "You can ask about studies, technology, science, and many other topics."
}

In [6]:
stop_words = set(stopwords.words('english'))


In [7]:
#CLEAN
def clean_text(text):
    text = text.lower()
    text = text.replace("?", "")
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text.strip()

In [8]:
#TOKENIZE
def tokenize_text(text):
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word not in stop_words]
    return tokens

In [9]:
#PREPROCESS
def preprocess(text):
    cleaned = clean_text(text)
    tokens = tokenize_text(cleaned)
    return " ".join(tokens)

In [10]:
questions = list(faq_data.keys())
answers = list(faq_data.values())

In [11]:
processed_questions = [preprocess(q) for q in questions]

In [12]:
vectorizer = TfidfVectorizer()
question_vectors = vectorizer.fit_transform(processed_questions)

In [ ]:
# GET_RESPONSE FUNCTION
def get_response(user_query):
    raw_query = user_query.lower().strip()

    # Exact match
    for q, a in faq_data.items():
        if raw_query.replace("?", "").strip() == q.lower().replace("?", "").strip():
            return a

    parts = re.split(r'[?.]|(?<=\w)\s+(?=[a-z])', raw_query)
    parts = [p.strip() for p in parts if p.strip()]

    responses = []

    for part in parts:
        user_query_processed = preprocess(part)
        user_vector = vectorizer.transform([user_query_processed])

        similarity = cosine_similarity(user_vector, question_vectors).flatten()

        index = similarity.argmax()
        score = similarity[index]

        if score < 0.2:
            continue
        else:
            responses.append(answers[index])

    if not responses:
        return "Sorry, I don't understand your question."

    return " ".join(responses)


# CHATBOT LOOP
print("🤖 FAQ Chatbot is ready! Type 'exit' to stop.\n")

while True:
    user_input = input("You: ")

    if user_input.lower() == "exit":
        print("Chatbot: Goodbye!")
        break

    response = get_response(user_input)
    print("Chatbot:", response)

🤖 FAQ Chatbot is ready! Type 'exit' to stop.

You: How are you?
Chatbot: I am doing well. Thanks for asking!
You:  how are you
Chatbot: I am doing well. Thanks for asking!
